# Stochastic Calculus from Zero · Episode 10: Brownian paths are strange

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. The slope that isn't · 4. Zoom in · 5. Infinite length · 6. History · 7. This episode's question · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 10,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)

## 2. Last episode's answer

$\mathrm{Cov}(W_2, W_5) = \min(2, 5) = 2$.

In [ ]:
out["prev"] = {"answer": 2}

## 3. The slope that isn't

One path on a fine grid of $2^{20}$ steps (about a million). Measure its slope over a window of length $h$, $(W_{t+h} - W_t)/h$, and average its size over the year. For a smooth path this would settle on the average size of the derivative as $h$ shrinks. Here $W_{t+h} - W_t$ is normal with standard deviation $\sqrt{h}$, so the slope is about $1/\sqrt{h}$: on average $\sqrt{2/(\pi h)}$, which grows without limit.

In [ ]:
rng = np.random.default_rng(seed)
brownian_paths(1, 1_000, 1.0, rng)                 # Episode 1's path comes first from this seed; skip it
FINE = 2 ** 20
_, Wf, _ = brownian_paths(1, FINE, 1.0, rng)
wf = Wf[0]
slopes = []
for k in (2 ** 17, 2 ** 13, 2 ** 10, 2 ** 6):
    h = k / FINE
    q = np.abs(np.diff(wf[::k])) / h
    slopes.append({"h": h, "one_over": round(1 / h), "slope": float(q.mean()), "theory": math.sqrt(2 / (math.pi * h))})
    assert abs(slopes[-1]["slope"] / slopes[-1]["theory"] - 1) < 0.1
out["slopes"] = {"rows": slopes, "s0": slopes[0]["slope"], "s3": slopes[-1]["slope"]}
pd.DataFrame(slopes)

## 4. Zoom in

Scaling (Episode 9) says the first $1/100$ of the path, stretched 100 times in time and $\sqrt{100} = 10$ times in height, is again a Brownian motion on $[0, 1]$. Zoom in on a smooth curve and it flattens into a straight line; zoom in on a Brownian path and it looks the same at every scale. Checked over 20,000 paths: the stretched window has the same variance at its end as the whole year.

In [ ]:
def window(frac, points=1_000):
    '''The first `frac` of the fine path, stretched to [0, 1] in time and by 1/sqrt(frac) in height.'''
    n = round(frac * FINE)
    idx = np.unique(np.linspace(0, n, min(points, n) + 1).round().astype(int))
    return (idx / n).round(5).tolist(), [(wf[idx] / math.sqrt(frac)).round(4).tolist()]

zoom = {}
for name, frac in (("whole", 1.0), ("hundredth", 0.01), ("ten_thousandth", 0.0001)):
    t_, p_ = window(frac)
    zoom[name] = {"t": t_, "path": p_, "frac": frac}
_, Wz, _ = brownian_paths(20_000, 1_000, 1.0, rng)
zoom["var_whole"] = float(Wz[:, -1].var())
zoom["var_window"] = float((Wz[:, 10] * 10).var())          # first 1/100, stretched by 10
assert abs(zoom["var_window"] - 1) < 0.05
zoom["ticks"] = [{"x": x, "label": lab} for x, lab in ((0, "start"), (0.5, "middle"), (1, "end of window"))]
# zeros: sign changes on the fine grid in the first 1/100 and 1/10,000 of the year, and in the whole year
sign = np.sign(wf[1:])
crossings = lambda n: int((np.diff(sign[:n]) != 0).sum())
zoom["zeros"] = {"whole": crossings(FINE), "hundredth": crossings(FINE // 100), "ten_thousandth": crossings(FINE // 10_000)}
out["zoom"] = zoom
{k: v for k, v in zoom.items() if k.startswith("var")}

## 5. Infinite length

The length of the path (the total distance travelled, ignoring direction) measured with $n$ steps is about $n \times \sqrt{2/(\pi n)} = \sqrt{2n/\pi}$: infinite in the limit. Series 3 met this as "infinitely wiggly". Its consequence here: the integrals of ordinary calculus, which add up $f \times \Delta W$ along a path of finite length, don't apply to $\int f\,dW$. Episode 13 builds a new integral.

In [ ]:
lengths = []
for n in (10, 1_000, FINE):
    idx = np.linspace(0, FINE, n + 1).round().astype(int)
    lengths.append({"steps": n, "length": float(np.abs(np.diff(wf[idx])).sum()), "theory": math.sqrt(2 * n / math.pi)})
assert abs(lengths[-1]["length"] / lengths[-1]["theory"] - 1) < 0.01
out["lengths"] = {"rows": lengths, "fine": lengths[-1]["length"], "fine_steps": FINE}
pd.DataFrame(lengths)

## 6. History

Raymond Paley, Norbert Wiener and Antoni Zygmund proved in 1933 (*Notes on random functions*, Mathematische Zeitschrift 37) that almost every Brownian path is nowhere differentiable.

In [ ]:
out["history"] = {"pwz": 1933}

## 7. This episode's question

Halve $h$ four times: $h \to h/16$, so the typical slope, about $1/\sqrt{h}$, grows by $\sqrt{16} = 4$.

In [ ]:
out["question"] = {"halvings": 4, "factor": math.sqrt(2 ** 4)}

## 8. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** For a normal $Z$ with mean 0 and variance $h$, show $E|Z| = \sqrt{2h/\pi}$.

<details><summary>Answer</summary>

$E|Z| = 2\int_0^\infty z \frac{1}{\sqrt{2\pi h}} e^{-z^2/2h}\,dz = \frac{2}{\sqrt{2\pi h}} \cdot h = \sqrt{2h/\pi}$.

</details>

**Exercise 2 (By hand).** If a path had finite length $L$, show that its sum of squared steps $\sum (\Delta W)^2$ would go to 0 as the steps shrink. Why does this rule Brownian motion out?

<details><summary>Answer</summary>

$\sum (\Delta W)^2 \le \max|\Delta W| \cdot \sum |\Delta W| \le \max|\Delta W| \cdot L \to 0$, since a continuous path has $\max|\Delta W| \to 0$. Brownian squared steps add up to $T$ (Episode 1), so its length can't be finite.

</details>

**Exercise 3 (By hand).** To make the first $1/10{,}000$ of the year look like a whole-year Brownian path, by how much must you stretch time, and height?

<details><summary>Answer</summary>

Time by 10,000 and height by $\sqrt{10{,}000} = 100$.

</details>

**Exercise 4 (In code).** Fit a line to log(average slope) against log(h) from Section 3. What slope do you expect?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
x = np.log([r["h"] for r in slopes]); y = np.log([r["slope"] for r in slopes])
b = np.polyfit(x, y, 1)[0]
assert -0.55 < b < -0.45
print(f"slope of the fit {b:.3f}: the typical slope grows like h^(-1/2)")

**Exercise 5 (In code).** Count how often the fine path changes sign in the first 1% of the year, and in the whole year. What does that say about the zeros of a Brownian path?

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
s = np.sign(wf[1:])                       # skip W_0 = 0, which has no sign
first = int((np.diff(s[: FINE // 100]) != 0).sum())
whole = int((np.diff(s) != 0).sum())
assert (first, whole) == (zoom["zeros"]["hundredth"], zoom["zeros"]["whole"])
print(f"sign changes: {first} in the first 1% of the year (an even spread would give about {whole // 100}), {whole} in the whole year")

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")